# ARENA3 런타임 B — B′ 자료 수집(공통 기반)

M4를 기본으로 선택 지점마다 SUMO 롤아웃으로 고르는 **B′**를 학습 날짜 64에피소드에서 돌리고,
선택 지점마다 관측·후보별 롤아웃 비용(전체·좌회전)을 남깁니다. Claude·Gemini 학습기가 이 자료로 배웁니다.
B′는 탐색 제어이고 강화학습이 아닙니다.

1. 런타임을 **CPU**로 고르고 **런타임 → 모두 실행**을 누르세요.
2. 순서: **실행 전 점검**(상태 저장·롤아웃을 해도 항상 M4를 고르면 일반 M4와 같은지, 5~10분) → 자료 64에피소드(약 12시간, 둘로 나눠 동시에).
3. 끊기면 다시 **모두 실행**을 누르세요. 끝난 에피소드는 건너뜁니다.

In [ ]:
from google.colab import drive
from pathlib import Path
import os, sys, json

drive.mount("/content/drive")
DRIVE = Path("/content/drive/MyDrive/FutureTraffic")
ARCHIVE = DRIVE / "FutureTraffic_ARENA3.zip"
A3 = DRIVE / "arena3"
WORKSPACE = Path("/content/FutureTraffic_ARENA3")
assert ARCHIVE.is_file(), f"묶음이 없습니다: {ARCHIVE}"
CPUS = os.cpu_count() or 1
print("CPU 개수:", CPUS, "| 결과 폴더:", A3)

In [ ]:
# ZIP 전체의 파일 목록·SHA-256을 확인한 뒤 검증된 전용 추출기를 사용합니다.
import hashlib, stat, types, zipfile
from pathlib import PurePosixPath

with zipfile.ZipFile(ARCHIVE) as archive:
    names = archive.namelist()
    assert len(names) == len(set(names)) and names.count("manifest.json") == 1, "ZIP 파일 목록 오류"
    package_manifest = json.loads(archive.read("manifest.json"))
    assert package_manifest.get("kind") == "WEIGHT1_PACKAGE_V1", "검증할 수 없는 ZIP 형식입니다"
    assert package_manifest.get("experiment") == "ARENA3_V1", "ARENA3 전용 ZIP이 아닙니다"
    assert package_manifest.get("source_models_included") is False, "모델·결과가 든 묶음은 쓰지 않습니다"
    expected = package_manifest["files"]
    assert expected and "manifest.json" not in expected and set(names) == set(expected) | {"manifest.json"}, "ZIP과 manifest 목록 불일치"
    for info in archive.infolist():
        name, path = info.filename, PurePosixPath(info.filename)
        assert path.parts and not path.is_absolute() and ".." not in path.parts and path.as_posix() == name, "잘못된 ZIP 경로"
        assert "\\" not in name and not info.is_dir() and not stat.S_ISLNK(info.external_attr >> 16), "잘못된 ZIP 파일 형식"
        if name != "manifest.json":
            assert hashlib.sha256(archive.read(name)).hexdigest() == expected[name], f"SHA-256 불일치: {name}"
    source_code = archive.read("src/explore_weight_package.py").decode("utf-8")

bootstrap = types.ModuleType("arena3_verified_package")
bootstrap.__file__ = str(WORKSPACE / "src/explore_weight_package.py")
exec(compile(source_code, bootstrap.__file__, "exec"), bootstrap.__dict__)
package_manifest = bootstrap.extract_verified(ARCHIVE, WORKSPACE)
print(f"검증·추출 완료: {len(package_manifest['files'])}개 파일 → {WORKSPACE} | 잠금 {package_manifest['lock_sha256'][:12]}")

In [ ]:
import subprocess, importlib.metadata

os.environ["FOCUSED_RULESET"] = "pet1"
os.environ["FOCUSED_NORTH_EXT"] = "240"
EXPECTED = {"numpy": "2.1.3", "torch": "2.11.0+cpu", "stable-baselines3": "2.9.0",
            "sumo-rl": "1.4.5", "eclipse-sumo": "1.27.1"}

def installed_versions():
    versions = {}
    for name in EXPECTED:
        try:
            versions[name] = importlib.metadata.version(name)
        except importlib.metadata.PackageNotFoundError:
            versions[name] = None
    return versions

before = installed_versions()
if before == EXPECTED:
    print("이전 실험과 패키지 버전이 같아 재설치를 건너뜁니다.")
else:
    print("이전 실험과 같은 패키지 버전을 준비합니다:", before)
    subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], cwd=WORKSPACE, check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "numpy==2.1.3", "stable-baselines3==2.9.0", "eclipse-sumo==1.27.1"],
                   cwd=WORKSPACE, check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "torch==2.11.0+cpu", "--index-url", "https://download.pytorch.org/whl/cpu"],
                   cwd=WORKSPACE, check=True)
after = installed_versions()
assert after == EXPECTED, f"패키지 버전이 이전 실험과 다릅니다(재현 점검이 실패할 수 있어 멈춥니다): {after}"
print("실행 패키지 검증 통과:", after)

In [ ]:
sys.path.insert(0, str(WORKSPACE / "src"))
import arena3_colab as colab
from IPython.display import display, Markdown

colab.run_live(["-m", "unittest", "discover", "-s", "tests", "-p", "test_arena3.py"], WORKSPACE)
print("실행 전 검사(SUMO 없이) 통과")

In [ ]:
# 1) 실행 전 점검. 통과 기록이 있으면 건너뜁니다.
BDATA = A3 / "bprime_data"
smoke = colab.read_json(BDATA / "smoke.json")
experiment = colab.read_json(BDATA / "experiment.json")
if smoke and experiment and smoke.get("fingerprint") == experiment.get("fingerprint") \
        and smoke.get("unchanged_by_saving") and smoke["restore_check"]["mismatched"] == 0:
    print("이전 점검 통과 기록이 있어 건너뜁니다.")
else:
    colab.run_live(["src/arena3_bprime.py", "smoke", "--out", str(BDATA), "--work", "/content/b3_smoke",
                    "--rollout-workers", str(CPUS)], WORKSPACE)
print(json.dumps(colab.read_json(BDATA / "smoke.json"), ensure_ascii=False, indent=1)[:4000])

In [ ]:
# 2) B′ 자료 64에피소드: 둘로 나눠 동시에. 5분마다 진행을 보여 줍니다.
LOGS = A3 / "logs" / "B"
half = max(1, CPUS // 2)
jobs = {f"bprime_data_{i}": ["src/arena3_bprime.py", "run", "--role", "data", "--out", str(BDATA),
                             "--work", f"/content/b3_{i}", "--shard", str(i), "--shards", "2",
                             "--rollout-workers", str(half)] for i in range(2)}

def describe(name):
    s = colab.read_json(BDATA / "status" / f"data_shard{name[-1]}.json")
    return None if not s else (f"{s['done']}/{s['total']} 끝, 실패 {len(s['failed'])}, "
                               f"복원 불일치 {s['restore_mismatched']}/{s['restore_checked']}")

pids = colab.launch(jobs, WORKSPACE, LOGS)
code = colab.watch(pids, LOGS, 300, describe)
assert code == 0, "B′ 자료 수집에서 오류가 났습니다. 위 로그를 저에게 보내 주세요."

In [ ]:
# 3) 자료 요약
rows = [colab.read_json(d / "metrics.json") for d in sorted((BDATA / "runs" / "BPRIME").iterdir()) if d.is_dir()]
rows = [m for m in rows if m]
points = sum(m["bprime_points"] for m in rows)
switched = sum(m["bprime_switched"] for m in rows)
cpu = sum(m["bprime_main_seconds"] + m["bprime_rollout_cpu_seconds"] for m in rows) / 3600
print(f"에피소드 {len(rows)}/64 | 선택 지점 {points}개 | M4에서 바꾼 지점 {switched}개({switched / max(1, points):.1%}) | {cpu:.1f} CPU-시간")